In [ ]:
## Fetch Affected CVES from National Vulnerbaility Database On the basis of keyword. Only Run to Inspect CVE's Related to a Specific Keyword, not to fetch actual device CVE. That is in hte second block
# import requests
# import json
# import os
# import time
# from datetime import datetime

# def fetch_cves_by_keyword(keyword="infusion pump", exact_match=False, start_index=0, results_per_page=2000):
#     """
#     Fetch CVEs from the NVD API filtered by keyword search.
#     - keywordExactMatch is a flag parameter with no value: include it when exact_match is True.
#     """
#     base_url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    

#     # Build params; let requests encode them
#     params = {
#         "keywordSearch": keyword,
#         "startIndex": start_index,
#         "resultsPerPage": results_per_page,
#     }
#     # Include the flag with no value when exact_match is requested
#     if exact_match:
#         # requests encodes empty-string value as just the key (…&keywordExactMatch)
#         params["keywordExactMatch"] = ""

#     time.sleep(6)  # respect NVD rate limiting

#     headers = {
#         "User-Agent": "CVE-Fetcher/1.0 (Research Purpose)"
#     }

#     try:
#         response = requests.get(base_url, params=params, headers=headers, timeout=60)
#         if response.status_code == 200:
#             return response.json()
#         elif response.status_code == 429:
#             print("Rate limit exceeded. Waiting 30 seconds before retrying...")
#             time.sleep(30)
#             return fetch_cves_by_keyword(keyword, exact_match, start_index, results_per_page)
#         else:
#             print(f"Error fetching data: {response.status_code}")
#             print(f"Response: {response.text}")
#             return None
#     except Exception as e:
#         print(f"Exception occurred: {e}")
#         time.sleep(30)
#         return fetch_cves_by_keyword(keyword, exact_match, start_index, results_per_page)

# def save_cve_to_file(cve_data):
#     cve_id = cve_data.get("id", "unknown")
#     filename = f"{cve_id}.json".replace('/', '_')
#     os.makedirs("cve_data", exist_ok=True)
#     file_path = os.path.join("cve_data", filename)
#     with open(file_path, 'w') as f:
#         json.dump(cve_data, f, indent=2)
#     return file_path

# def main():
#     os.makedirs("cve_data", exist_ok=True)
#     log_file = os.path.join("cve_data", "fetch_log.txt")

#     start_time = datetime.now()

#     start_index = 0
#     results_per_page = 2000
#     total_saved = 0

#     progress_file = "progress.json"
#     if os.path.exists(progress_file):
#         with open(progress_file, 'r') as f:
#             progress_data = json.load(f)
#             start_index = progress_data.get("next_start_index", 0)
#             total_saved = progress_data.get("total_saved", 0)
#             print(f"Resuming from index {start_index}, already saved {total_saved} CVEs")

#     keyword = "infusion pump"
#     exact_match = False  # include bare flag keywordExactMatch

#     while True:
#         print(f"Fetching CVEs (keyword='{keyword}', exact={exact_match}) from index {start_index}...")
#         response = fetch_cves_by_keyword(
#             keyword=keyword,
#             exact_match=exact_match,
#             start_index=start_index,
#             results_per_page=results_per_page
#         )

#         if not response:
#             print("Failed to fetch CVEs, retrying in 60 seconds...")
#             time.sleep(60)
#             continue

#         total_results = response.get("totalResults", 0)
#         results_per_page = response.get("resultsPerPage", results_per_page)
#         vulnerabilities = response.get("vulnerabilities", [])
#         batch_count = len(vulnerabilities)

#         if batch_count == 0:
#             print("No more CVEs to fetch.")
#             break

#         print(f"Total CVEs available matching keyword: {total_results}")
#         print(f"Processing {batch_count} vulnerabilities...")

#         with open(log_file, 'a') as log:
#             log.write(f"[{datetime.now()}] Fetching batch starting at index {start_index}, containing {batch_count} CVEs\n")

#         current_batch_saved = 0
#         for vuln in vulnerabilities:
#             cve_item = vuln.get("cve", {})
            
#             save_cve_to_file(cve_item)
#             current_batch_saved += 1
#             if current_batch_saved % 100 == 0:
#                 print(f"Saved {current_batch_saved}/{batch_count} CVEs in current batch")

#         total_saved += current_batch_saved
#         next_start_index = start_index + batch_count

#         with open(progress_file, 'w') as f:
#             json.dump({
#                 "next_start_index": next_start_index,
#                 "total_saved": total_saved,
#                 "total_results": total_results,
#                 "keyword": keyword,
#                 "exact_match": exact_match,
#                 "last_update": datetime.now().isoformat()
#             }, f, indent=2)

#         if next_start_index >= total_results:
#             print(f"Completed fetching all {total_results} CVEs for keyword '{keyword}'.")
#             break

#         start_index = next_start_index

#         progress_percentage = (total_saved / total_results) * 100 if total_results else 0
#         elapsed_time = (datetime.now() - start_time).total_seconds() / 60
#         estimated_total_time = (elapsed_time / progress_percentage) * 100 if progress_percentage > 0 else 0
#         estimated_remaining_time = estimated_total_time - elapsed_time

#         print(f"Overall progress: {total_saved}/{total_results} CVEs ({progress_percentage:.2f}%)")
#         print(f"Elapsed time: {elapsed_time:.2f} minutes")
#         print(f"Estimated remaining time: {estimated_remaining_time:.2f} minutes")
#         print("-----------------------------------------------")

#     end_time = datetime.now()
#     total_time = (end_time - start_time).total_seconds() / 60
#     print(f"Completed saving {total_saved} CVEs in {total_time:.2f} minutes.")
#     with open(log_file, 'a') as log:
#         log.write(f"[{datetime.now()}] Completed fetching all {total_saved} CVEs in {total_time:.2f} minutes.\n")

# if __name__ == "__main__":
#     main()


In [3]:
import json
import os
import shutil
import requests
import time
from urllib.parse import quote

def fetch_cves_by_cpe(cpe_name, results_per_page=2000, delay=6):
    """
    Fetch CVEs from NVD API based on CPE name
    
    Args:
        cpe_name: CPE identifier to search for
        results_per_page: Number of results per API call (max 2000)
        delay: Delay between API calls in seconds (NVD rate limit is 5 seconds)
    
    Returns:
        List of CVE entries
    """
    base_url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    encoded_cpe = quote(cpe_name, safe='')
    
    all_vulnerabilities = []
    start_index = 0
    
    print(f"Fetching CVEs for CPE: {cpe_name}")
    print(f"Using delay of {delay} seconds between requests...")
    
    while True:
        # Construct URL with parameters
        url = f"{base_url}?cpeName={encoded_cpe}&resultsPerPage={results_per_page}&startIndex={start_index}"
        
        print(f"Fetching results starting from index {start_index}...")
        
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            
            data = response.json()
            vulnerabilities = data.get('vulnerabilities', [])
            total_results = data.get('totalResults', 0)
            
            print(f"Retrieved {len(vulnerabilities)} CVEs (Total: {total_results})")
            
            if not vulnerabilities:
                break
                
            all_vulnerabilities.extend(vulnerabilities)
            
            # Check if we've retrieved all results
            if len(all_vulnerabilities) >= total_results:
                break
                
            start_index += results_per_page
            
            # Respect rate limits
            print(f"Waiting {delay} seconds before next request...")
            time.sleep(delay)
            
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON response: {e}")
            break
    
    print(f"Total CVEs fetched: {len(all_vulnerabilities)}")
    return all_vulnerabilities

def process_and_save_cves(cpe_name):
    """
    Fetch CVEs by CPE and organize them into accepted/rejected
    """
    # Directories
    cve_dir = "cve_data"
    rejected_dir = "rejected_cve_data"
    
    # Create directories if they don't exist
    os.makedirs(cve_dir, exist_ok=True)
    os.makedirs(rejected_dir, exist_ok=True)
    
    # Fetch CVEs from API
    vulnerabilities = fetch_cves_by_cpe(cpe_name)
    
    if not vulnerabilities:
        print("No CVEs found for the specified CPE.")
        return
    
    # Counters
    total_cves = len(vulnerabilities)
    rejected_count = 0
    accepted_count = 0
    
    print("\nProcessing and saving CVEs...")
    
    # Process each CVE
    for vuln in vulnerabilities:
        cve_data = vuln.get('cve', {})
        cve_id = cve_data.get('id', 'UNKNOWN')
        vuln_status = cve_data.get('vulnStatus', '')
        
        # Create filename
        filename = f"{cve_id}.json"
        
        # Determine destination directory based on status
        if vuln_status == "Rejected":
            file_path = os.path.join(rejected_dir, filename)
            rejected_count += 1
        else:
            file_path = os.path.join(cve_dir, filename)
            accepted_count += 1
        
        # Save CVE data
        try:
            with open(file_path, "w", encoding="utf-8") as f:
                json.dump(cve_data, f, indent=2, ensure_ascii=False)
        except Exception as e:
            print(f"Error saving {cve_id}: {e}")
    
    # Print results
    print(f"\n=== PROCESSING COMPLETE ===")
    print(f"Total CVEs processed: {total_cves}")
    print(f"Accepted CVEs saved to '{cve_dir}': {accepted_count}")
    print(f"Rejected CVEs saved to '{rejected_dir}': {rejected_count}")
    
    return {
        'total': total_cves,
        'accepted': accepted_count,
        'rejected': rejected_count
    }

def main():
    # CPE for Baxter SIGMA Spectrum Infusion System
    cpe_name = "cpe:2.3:h:baxter:sigma_spectrum_infusion_system:-:*:*:*:*:*:*:*"
    
    print("CVE Fetcher by CPE")
    print("=" * 50)
    
    try:
        results = process_and_save_cves(cpe_name)
        
        # Additional summary
        print(f"\nFiles can be found in:")
        print(f"  - Accepted CVEs: ./cve_data/")
        print(f"  - Rejected CVEs: ./rejected_cve_data/")
        
    except KeyboardInterrupt:
        print("\nOperation cancelled by user.")
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()

CVE Fetcher by CPE
Fetching CVEs for CPE: cpe:2.3:h:baxter:sigma_spectrum_infusion_system:-:*:*:*:*:*:*:*
Using delay of 6 seconds between requests...
Fetching results starting from index 0...
Retrieved 10 CVEs (Total: 10)
Total CVEs fetched: 10

Processing and saving CVEs...

=== PROCESSING COMPLETE ===
Total CVEs processed: 10
Accepted CVEs saved to 'cve_data': 10
Rejected CVEs saved to 'rejected_cve_data': 0

Files can be found in:
  - Accepted CVEs: ./cve_data/
  - Rejected CVEs: ./rejected_cve_data/


# Remove CVE entries which are Rejected

In [4]:
import json
import os
import shutil  # For moving files

# Directories
cve_dir = "cve_data"
rejected_dir = "rejected_cve_data"  # Directory for rejected CVEs

# Create directory for rejected CVEs if it doesn't exist
os.makedirs(rejected_dir, exist_ok=True)

# Counters
removed_count = 0
total_files = 0

# Process CVE files
for filename in os.listdir(cve_dir):
    if filename.endswith(".json"):
        total_files += 1
        file_path = os.path.join(cve_dir, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            cve_entry = json.load(f)

        if cve_entry.get("vulnStatus") == "Rejected":
            removed_count += 1
            shutil.move(file_path, os.path.join(rejected_dir, filename))  # Move rejected CVE

# Print results
print(f"Total files processed: {total_files}")
print(f"Rejected CVEs moved: {removed_count}")
print(f"Remaining CVEs in original folder: {total_files - removed_count}")


Total files processed: 10
Rejected CVEs moved: 0
Remaining CVEs in original folder: 10


In [ ]:
import os, json, re
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side, GradientFill
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule

json_dir = "cve_data"

# ── helpers ────────────────────────────────────────────────────────────────────
def parse_cpe(cpe_str):
    """Parse CPE 2.3 URI into (vendor, product, version)."""
    parts = cpe_str.split(":")
    vendor  = parts[3] if len(parts) > 3 else "N/A"
    product = parts[4] if len(parts) > 4 else "N/A"
    version = parts[5] if len(parts) > 5 else "N/A"
    return vendor.replace("_", " ").title(), product.replace("_", " ").title(), version

def extract_cvss(metrics):
    m = {}
    if "cvssMetricV31" in metrics:
        raw = metrics["cvssMetricV31"][0]
        d = raw["cvssData"]
        m.update({
            "CVSS Version": "3.1",
            "baseScore": d.get("baseScore"), "baseSeverity": d.get("baseSeverity"),
            "attackVector": d.get("attackVector"), "attackComplexity": d.get("attackComplexity"),
            "privilegesRequired": d.get("privilegesRequired"), "userInteraction": d.get("userInteraction"),
            "scope": d.get("scope"), "confidentialityImpact": d.get("confidentialityImpact"),
            "integrityImpact": d.get("integrityImpact"), "availabilityImpact": d.get("availabilityImpact"),
            "exploitabilityScore": raw.get("exploitabilityScore"), "impactScore": raw.get("impactScore"),
            "vectorString": d.get("vectorString"),
        })
    elif "cvssMetricV30" in metrics:
        raw = metrics["cvssMetricV30"][0]
        d = raw["cvssData"]
        m.update({
            "CVSS Version": "3.0",
            "baseScore": d.get("baseScore"), "baseSeverity": d.get("baseSeverity"),
            "attackVector": d.get("attackVector"), "attackComplexity": d.get("attackComplexity"),
            "privilegesRequired": d.get("privilegesRequired"), "userInteraction": d.get("userInteraction"),
            "scope": d.get("scope"), "confidentialityImpact": d.get("confidentialityImpact"),
            "integrityImpact": d.get("integrityImpact"), "availabilityImpact": d.get("availabilityImpact"),
            "exploitabilityScore": raw.get("exploitabilityScore"), "impactScore": raw.get("impactScore"),
            "vectorString": d.get("vectorString"),
        })
    elif "cvssMetricV2" in metrics:
        raw = metrics["cvssMetricV2"][0]
        d = raw["cvssData"]
        m.update({
            "CVSS Version": "2.0",
            "baseScore": d.get("baseScore"), "baseSeverity": raw.get("baseSeverity"),
            "attackVector": d.get("accessVector"), "attackComplexity": d.get("accessComplexity"),
            "privilegesRequired": d.get("authentication"),
            "userInteraction": "REQUIRED" if raw.get("userInteractionRequired") else "NONE",
            "scope": "N/A",
            "confidentialityImpact": d.get("confidentialityImpact"),
            "integrityImpact": d.get("integrityImpact"),
            "availabilityImpact": d.get("availabilityImpact"),
            "exploitabilityScore": raw.get("exploitabilityScore"), "impactScore": raw.get("impactScore"),
            "vectorString": d.get("vectorString"),
        })
    return m

# ── parse all files ────────────────────────────────────────────────────────────
main_rows, config_rows, ref_rows = [], [], []

for fname in sorted(os.listdir(json_dir)):
    if not fname.endswith(".json"):
        continue
    data = json.load(open(os.path.join(json_dir, fname), encoding="utf-8"))
    cve_id = data["id"]

    # English description
    desc = next((d["value"] for d in data.get("descriptions", []) if d.get("lang") == "en"), "N/A")

    # Weaknesses
    weak_primary = weak_secondary = "N/A"
    for w in data.get("weaknesses", []):
        val = next((d["value"] for d in w.get("description", []) if d.get("lang") == "en"), None)
        if val:
            if w.get("type") == "Primary":
                weak_primary = val
            else:
                weak_secondary = val

    # CVSS
    cvss = extract_cvss(data.get("metrics", {}))

    main_rows.append({
        "CVE ID": cve_id,
        "Published": data.get("published", "")[:10],
        "Last Modified": data.get("lastModified", "")[:10],
        "Status": data.get("vulnStatus", "N/A"),
        "Source": data.get("sourceIdentifier", "N/A"),
        "Description": desc,
        "Primary Weakness (CWE)": weak_primary,
        "Secondary Weakness (CWE)": weak_secondary,
        "CVSS Version": cvss.get("CVSS Version"),
        "Base Score": cvss.get("baseScore"),
        "Severity": cvss.get("baseSeverity"),
        "Attack Vector": cvss.get("attackVector"),
        "Attack Complexity": cvss.get("attackComplexity"),
        "Privileges Required": cvss.get("privilegesRequired"),
        "User Interaction": cvss.get("userInteraction"),
        "Scope": cvss.get("scope"),
        "Confidentiality Impact": cvss.get("confidentialityImpact"),
        "Integrity Impact": cvss.get("integrityImpact"),
        "Availability Impact": cvss.get("availabilityImpact"),
        "Exploitability Score": cvss.get("exploitabilityScore"),
        "Impact Score": cvss.get("impactScore"),
        "Vector String": cvss.get("vectorString"),
    })

    # Configurations → affected devices
    for cfg in data.get("configurations", []):
        for node in cfg.get("nodes", []):
            for cpe in node.get("cpeMatch", []):
                vendor, product, version = parse_cpe(cpe.get("criteria", ""))
                config_rows.append({
                    "CVE ID": cve_id,
                    "Vulnerable": cpe.get("vulnerable", False),
                    "Vendor": vendor,
                    "Product / Component": product,
                    "Version": version,
                    "CPE Criteria": cpe.get("criteria", "N/A"),
                })

    # References
    seen_urls = set()
    for ref in data.get("references", []):
        url = ref.get("url", "")
        if url in seen_urls:
            continue
        seen_urls.add(url)
        ref_rows.append({
            "CVE ID": cve_id,
            "URL": url,
            "Tags": ", ".join(ref.get("tags", [])),
        })

df_main   = pd.DataFrame(main_rows)
df_config = pd.DataFrame(config_rows)
df_ref    = pd.DataFrame(ref_rows)

# ── build Excel ────────────────────────────────────────────────────────────────
wb = Workbook()

# colour palette
C_HEADER_BG  = "1A2B4A"  # dark navy
C_HEADER_FG  = "FFFFFF"
C_ACCENT     = "E63946"  # red accent
C_ROW_ALT    = "F0F4F8"
C_BORDER     = "C8D0DC"

SEVERITY_COLORS = {
    "CRITICAL": "7B0000",
    "HIGH":     "C0392B",
    "MEDIUM":   "D4860A",
    "LOW":      "2E7D32",
    "NONE":     "558B2F",
}
SEV_FG = {k: "FFFFFF" for k in SEVERITY_COLORS}
SEV_FG["LOW"] = "FFFFFF"

thin = Side(border_style="thin", color=C_BORDER)
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def style_header(cell, bg=C_HEADER_BG, fg=C_HEADER_FG):
    cell.font = Font(bold=True, color=fg, name="Calibri", size=10)
    cell.fill = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = border

def style_data(cell, alt=False):
    cell.font = Font(name="Calibri", size=10)
    if alt:
        cell.fill = PatternFill("solid", fgColor=C_ROW_ALT)
    cell.alignment = Alignment(vertical="top", wrap_text=True)
    cell.border = border

def autofit(ws, min_w=10, max_w=50):
    for col in ws.columns:
        length = max((len(str(c.value or "")) for c in col), default=0)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(length + 2, min_w), max_w)

def write_sheet(ws, df, col_widths=None):
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    headers = list(df.columns)
    for ci, h in enumerate(headers, 1):
        cell = ws.cell(row=1, column=ci, value=h)
        style_header(cell)
    for ri, row in enumerate(df.itertuples(index=False), 2):
        alt = ri % 2 == 0
        for ci, val in enumerate(row, 1):
            cell = ws.cell(row=ri, column=ci, value=val)
            style_data(cell, alt)
    ws.row_dimensions[1].height = 28
    if col_widths:
        for col, w in col_widths.items():
            ws.column_dimensions[col].width = w
    else:
        autofit(ws)

# ── Sheet 1: CVE Summary ───────────────────────────────────────────────────────
ws1 = wb.active
ws1.title = "CVE Summary"
write_sheet(ws1, df_main, col_widths={
    "A": 18, "B": 12, "C": 14, "D": 14, "E": 28, "F": 60,
    "G": 20, "H": 22, "I": 10, "J": 10, "K": 10, "L": 15,
    "M": 16, "N": 18, "O": 16, "P": 10, "Q": 18, "R": 16,
    "S": 18, "T": 16, "U": 16, "V": 35,
})

# Colour severity cells (col K = "Severity")
sev_col_idx = df_main.columns.get_loc("Severity") + 1
for ri in range(2, len(df_main) + 2):
    cell = ws1.cell(row=ri, column=sev_col_idx)
    sev = str(cell.value or "").upper()
    if sev in SEVERITY_COLORS:
        cell.fill = PatternFill("solid", fgColor=SEVERITY_COLORS[sev])
        cell.font = Font(bold=True, color="FFFFFF", name="Calibri", size=10)

# Colour-scale Base Score (col J)
score_col = get_column_letter(df_main.columns.get_loc("Base Score") + 1)
score_range = f"{score_col}2:{score_col}{len(df_main)+1}"
ws1.conditional_formatting.add(score_range,
    ColorScaleRule(start_type="num", start_value=0, start_color="63BE7B",
                   mid_type="num",   mid_value=5,   mid_color="FFEB84",
                   end_type="num",   end_value=10,  end_color="F8696B"))

# ── Sheet 2: Affected Devices / Components ─────────────────────────────────────
ws2 = wb.create_sheet("Affected Components")
write_sheet(ws2, df_config, col_widths={"A": 18, "B": 12, "C": 20, "D": 30, "E": 12, "F": 55})

# highlight vulnerable=True rows
for ri in range(2, len(df_config) + 2):
    cell = ws2.cell(row=ri, column=2)  # "Vulnerable"
    if cell.value is True:
        cell.value = "YES"
        cell.fill = PatternFill("solid", fgColor="FDECEA")
        cell.font = Font(bold=True, color="C0392B", name="Calibri", size=10)
    else:
        cell.value = "no"

# ── Sheet 3: References ────────────────────────────────────────────────────────
ws3 = wb.create_sheet("References")
write_sheet(ws3, df_ref, col_widths={"A": 18, "B": 65, "C": 40})

# ── Sheet 4: Analytics Summary ─────────────────────────────────────────────────
ws4 = wb.create_sheet("Analytics")
ws4.sheet_view.showGridLines = False

def h1(ws, row, col, text):
    c = ws.cell(row=row, column=col, value=text)
    c.font = Font(bold=True, color=C_HEADER_FG, name="Calibri", size=12)
    c.fill = PatternFill("solid", fgColor=C_HEADER_BG)
    c.alignment = Alignment(horizontal="left", vertical="center")
    c.border = border
    return c

def kv(ws, row, lbl, val, col=1):
    lc = ws.cell(row=row, column=col, value=lbl)
    lc.font = Font(bold=True, name="Calibri", size=10)
    lc.fill = PatternFill("solid", fgColor="E8EDF3")
    lc.border = border
    lc.alignment = Alignment(horizontal="left", vertical="center")
    vc = ws.cell(row=row, column=col+1, value=val)
    vc.font = Font(name="Calibri", size=10)
    vc.border = border
    vc.alignment = Alignment(horizontal="center", vertical="center")
    return vc

# Overview block
h1(ws4, 1, 1, "📊  CVE Dataset — Analytics Overview")
ws4.merge_cells("A1:F1")
ws4.row_dimensions[1].height = 30

kv(ws4, 3, "Total CVEs", len(df_main))
kv(ws4, 4, "Total Affected Components", len(df_config[df_config["Vulnerable"] == True]))
kv(ws4, 5, "Total References", len(df_ref))
kv(ws4, 6, "Date Range (Published)",
   f"{df_main['Published'].min()}  →  {df_main['Published'].max()}")

# Severity breakdown
sev_counts = df_main["Severity"].value_counts()
h1(ws4, 9, 1, "Severity Breakdown")
ws4.merge_cells("A9:B9")
for i, (sev, cnt) in enumerate(sev_counts.items(), 10):
    c_lbl = ws4.cell(row=i, column=1, value=sev)
    c_cnt = ws4.cell(row=i, column=2, value=cnt)
    bg = SEVERITY_COLORS.get(str(sev).upper(), "888888")
    for c in (c_lbl, c_cnt):
        c.fill = PatternFill("solid", fgColor=bg)
        c.font = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
        c.border = border
        c.alignment = Alignment(horizontal="center", vertical="center")

# Attack Vector breakdown
av_counts = df_main["Attack Vector"].value_counts()
h1(ws4, 9, 4, "Attack Vector Breakdown")
ws4.merge_cells("D9:E9")
for i, (av, cnt) in enumerate(av_counts.items(), 10):
    c_lbl = ws4.cell(row=i, column=4, value=av)
    c_cnt = ws4.cell(row=i, column=5, value=cnt)
    for c in (c_lbl, c_cnt):
        c.fill = PatternFill("solid", fgColor="2C3E50")
        c.font = Font(color="FFFFFF", name="Calibri", size=10, bold=True)
        c.border = border
        c.alignment = Alignment(horizontal="center", vertical="center")

# Top affected vendors
vendor_counts = df_config[df_config["Vulnerable"] == True]["Vendor"].value_counts().head(10)
h1(ws4, 20, 1, "Top Affected Vendors (vulnerable components)")
ws4.merge_cells("A20:C20")
for i, (v, cnt) in enumerate(vendor_counts.items(), 21):
    c_lbl = ws4.cell(row=i, column=1, value=v)
    c_cnt = ws4.cell(row=i, column=2, value=cnt)
    c_lbl.font = Font(name="Calibri", size=10)
    c_cnt.font = Font(name="Calibri", size=10, bold=True)
    c_cnt.fill = PatternFill("solid", fgColor="DDEEFF")
    for c in (c_lbl, c_cnt):
        c.border = border
        c.alignment = Alignment(horizontal="left", vertical="center")

# Avg score table
h1(ws4, 20, 4, "Score Statistics")
ws4.merge_cells("D20:E20")
scores = df_main["Base Score"].dropna()
for i, (lbl, val) in enumerate([
    ("Average Base Score", round(scores.mean(), 2)),
    ("Max Base Score",     scores.max()),
    ("Min Base Score",     scores.min()),
    ("Median Base Score",  scores.median()),
], 21):
    ws4.cell(row=i, column=4, value=lbl).border = border
    ws4.cell(row=i, column=4).font = Font(name="Calibri", size=10)
    ws4.cell(row=i, column=4).alignment = Alignment(horizontal="left")
    c = ws4.cell(row=i, column=5, value=val)
    c.border = border
    c.font = Font(bold=True, name="Calibri", size=10)
    c.alignment = Alignment(horizontal="center")

for col, w in [("A",28),("B",12),("C",12),("D",32),("E",12)]:
    ws4.column_dimensions[col].width = w

# ── save ───────────────────────────────────────────────────────────────────────
out_path = "cve_data.xlsx"
wb.save(out_path)
print("Saved:", out_path)
print("Main rows:", len(df_main))
print("Config rows:", len(df_config))
print("Ref rows:", len(df_ref))

# Export JSON summary for dashboard
summary = {
    "cves": df_main.to_dict(orient="records"),
    "components": df_config.assign(Vulnerable=df_config["Vulnerable"].astype(str)).to_dict(orient="records"),
    "references": df_ref.to_dict(orient="records"),
}
# json.dump(summary, open("cve_summary.json", "w"), indent=2)
print("JSON summary saved.")

Saved: cve_data.xlsx
Main rows: 10
Config rows: 46
Ref rows: 10
JSON summary saved.


# Tabulating the CVEs entries in an excel sheet for Data Curation

In [3]:
import os
import json
import pandas as pd

# Define the directory containing JSON files
json_dir = "cve_data"

# List to store extracted CVE data
cve_data = []

# Iterate over all JSON files in the directory
for filename in os.listdir(json_dir):
    if filename.endswith(".json"):  # Process only JSON files
        file_path = os.path.join(json_dir, filename)
        
        # Open and read the JSON file
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
            
        # Extract the CVE ID
        cve_id = data.get("id")
        
        # Extract English description
        description = "N/A"
        descriptions = data.get("descriptions", [])
        for desc in descriptions:
            if desc.get("lang") == "en":
                description = desc.get("value", "N/A")
                break
        
        # Extract weaknesses
        weaknesses = []
        for weakness in data.get("weaknesses", []):
            for desc in weakness.get("description", []):
                if desc.get("lang") == "en":
                    weaknesses.append(desc.get("value", ""))
                    break
        
        # Join multiple weaknesses with a semicolon if present
        weakness_string = "; ".join(weaknesses) if weaknesses else "N/A"
        
        # Initialize metrics with default values
        metrics = {
            "CVE ID": cve_id,
            "Description": description,
            "Weaknesses": weakness_string,
            "baseScore": None,
            "baseSeverity": None,
            "attackVector": None,
            "attackComplexity": None,
            "privilegesRequired": None,
            "userInteraction": None,
            "scope": None,
            "confidentialityImpact": None,
            "integrityImpact": None,
            "availabilityImpact": None,
            "exploitabilityScore": None,
            "CVSS Version": None
        }
        
        # Extract metrics based on available CVSS versions
        # Priority: V4.0 > V3.1 > V3.0 > V2.0
        cvss_metrics = data.get("metrics", {})
        
        # # Try to extract CVSS v4.0 metrics
        # if "cvssMetricV40" in cvss_metrics:
        #     cvss_v4 = cvss_metrics["cvssMetricV40"][0]
        #     cvss_data = cvss_v4.get("cvssData", {})
            
        #     metrics["baseScore"] = cvss_data.get("baseScore")
        #     metrics["baseSeverity"] = cvss_data.get("baseSeverity")
        #     metrics["attackVector"] = cvss_data.get("attackVector")
        #     metrics["attackComplexity"] = cvss_data.get("attackComplexity")
        #     metrics["privilegesRequired"] = cvss_data.get("privilegesRequired")
        #     metrics["userInteraction"] = cvss_data.get("userInteraction")
        #     metrics["scope"] = "N/A"  # Not directly available in v4.0
        #     metrics["confidentialityImpact"] = cvss_data.get("vulnConfidentialityImpact")
        #     metrics["integrityImpact"] = cvss_data.get("vulnIntegrityImpact")
        #     metrics["availabilityImpact"] = cvss_data.get("vulnAvailabilityImpact")
        #     metrics["exploitabilityScore"] = cvss_v4.get("exploitabilityScore")
        #     metrics["CVSS Version"] = "4.0"
            
        # Try to extract CVSS v3.1 metrics
        if "cvssMetricV31" in cvss_metrics:
            cvss_v31 = cvss_metrics["cvssMetricV31"][0]
            cvss_data = cvss_v31.get("cvssData", {})
            
            metrics["baseScore"] = cvss_data.get("baseScore")
            metrics["baseSeverity"] = cvss_data.get("baseSeverity")
            metrics["attackVector"] = cvss_data.get("attackVector")
            metrics["attackComplexity"] = cvss_data.get("attackComplexity")
            metrics["privilegesRequired"] = cvss_data.get("privilegesRequired")
            metrics["userInteraction"] = cvss_data.get("userInteraction")
            metrics["scope"] = cvss_data.get("scope")
            metrics["confidentialityImpact"] = cvss_data.get("confidentialityImpact")
            metrics["integrityImpact"] = cvss_data.get("integrityImpact")
            metrics["availabilityImpact"] = cvss_data.get("availabilityImpact")
            metrics["exploitabilityScore"] = cvss_v31.get("exploitabilityScore")
            metrics["CVSS Version"] = "3.1"
            
        # Try to extract CVSS v3.0 metrics
        elif "cvssMetricV30" in cvss_metrics:
            cvss_v30 = cvss_metrics["cvssMetricV30"][0]
            cvss_data = cvss_v30.get("cvssData", {})
            
            metrics["baseScore"] = cvss_data.get("baseScore")
            metrics["baseSeverity"] = cvss_data.get("baseSeverity")
            metrics["attackVector"] = cvss_data.get("attackVector")
            metrics["attackComplexity"] = cvss_data.get("attackComplexity")
            metrics["privilegesRequired"] = cvss_data.get("privilegesRequired")
            metrics["userInteraction"] = cvss_data.get("userInteraction")
            metrics["scope"] = cvss_data.get("scope")
            metrics["confidentialityImpact"] = cvss_data.get("confidentialityImpact")
            metrics["integrityImpact"] = cvss_data.get("integrityImpact")
            metrics["availabilityImpact"] = cvss_data.get("availabilityImpact")
            metrics["exploitabilityScore"] = cvss_v30.get("exploitabilityScore")
            metrics["CVSS Version"] = "3.0"
            
        # Try to extract CVSS v2.0 metrics
        elif "cvssMetricV2" in cvss_metrics:
            cvss_v2 = cvss_metrics["cvssMetricV2"][0]
            cvss_data = cvss_v2.get("cvssData", {})
            
            metrics["baseScore"] = cvss_data.get("baseScore")
            metrics["baseSeverity"] = cvss_v2.get("baseSeverity")  # Note: baseSeverity is outside cvssData in v2
            metrics["attackVector"] = cvss_data.get("accessVector")  # Different name in v2
            metrics["attackComplexity"] = cvss_data.get("accessComplexity")  # Different name in v2
            metrics["privilegesRequired"] = cvss_data.get("authentication")  # Different name in v2
            metrics["userInteraction"] = "REQUIRED" if cvss_v2.get("userInteractionRequired") else "NONE"
            metrics["scope"] = "N/A"  # Not available in v2
            metrics["confidentialityImpact"] = cvss_data.get("confidentialityImpact")
            metrics["integrityImpact"] = cvss_data.get("integrityImpact")
            metrics["availabilityImpact"] = cvss_data.get("availabilityImpact")
            metrics["exploitabilityScore"] = cvss_v2.get("exploitabilityScore")
            metrics["CVSS Version"] = "2.0"
        
        # Add the extracted metrics to our list
        cve_data.append(metrics)

# Create a DataFrame from the extracted data
df = pd.DataFrame(cve_data)
df


,CVE ID,Description,Weaknesses,baseScore,baseSeverity,attackVector,attackComplexity,privilegesRequired,userInteraction,scope,confidentialityImpact,integrityImpact,availabilityImpact,exploitabilityScore,CVSS Version
0,CVE-2014-5433,An unauthenticated remote attacker may be able...,CWE-312; CWE-255,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,HIGH,3.9,3.0
1,CVE-2020-12040,Sigma Spectrum Infusion System v's6.x (model 3...,CWE-319; CWE-319,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,HIGH,3.9,3.1
2,CVE-2020-12047,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-259; CWE-798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,HIGH,3.9,3.1
3,CVE-2014-5431,Baxter SIGMA Spectrum Infusion System version ...,CWE-259; CWE-798,6.8,MEDIUM,PHYSICAL,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,HIGH,0.9,3.0
4,CVE-2014-5434,Baxter SIGMA Spectrum Infusion System version ...,CWE-259; CWE-798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,HIGH,3.9,3.0
5,CVE-2020-12045,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-259; CWE-798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,HIGH,3.9,3.1
6,CVE-2020-12039,Baxter Sigma Spectrum Infusion Pumps Sigma Spe...,CWE-259; CWE-798,2.4,LOW,PHYSICAL,LOW,NONE,NONE,UNCHANGED,LOW,NONE,NONE,0.9,3.1
7,CVE-2014-5432,Baxter SIGMA Spectrum Infusion System version ...,CWE-592; CWE-287,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,HIGH,3.9,3.0
8,CVE-2020-12041,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-732; CWE-732,9.4,CRITICAL,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,LOW,HIGH,3.9,3.1
9,CVE-2020-12043,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-672; CWE-672,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,UNCHANGED,HIGH,HIGH,HIGH,3.9,3.1


# Filter Rows With CWE as N/A

In [4]:

df = df.drop(columns=["scope"])
nan_rows_df = df[df.isna().any(axis=1)]  # Select rows where any column has NaN
nan_rows_df
df["Weaknesses"] = df["Weaknesses"].replace(["N/A","<NA>",""], pd.NA)
na_count = df["Weaknesses"].isna().sum()
print(na_count)

0


In [5]:

def clean_cwe_column(df):
    df['Weaknesses'] = df['Weaknesses'].astype(str)  # Ensure all values are strings
    
    # Split, remove duplicates, and filter out unwanted entries
    def process_weaknesses(entry):
        items = set(entry.split('; '))
        items.discard('NVD-CWE-noinfo')
        items.discard('NVD-CWE-Other')
        return '; '.join(sorted(items)) if items else ''
    
    df['Weaknesses'] = df['Weaknesses'].apply(process_weaknesses)
    
    return df


# Apply the function
df = clean_cwe_column(df)

In [6]:


# Store the total number of rows before filtering
total_rows_before = df.shape[0]

# Remove rows where 'Weaknesses' is NaN
df_filtered = df.dropna(subset=["Weaknesses"])

# Store the total number of rows after filtering
total_rows_after = df_filtered.shape[0]

# Calculate and print the number of rows removed
rows_removed = total_rows_before - total_rows_after
print(f"Total rows removed: {rows_removed}")




Total rows removed: 0


In [7]:
# Save to an Excel file
output_file = "cve_data.xlsx"
df_filtered.to_excel(output_file, index=False)

print(f"Extracted {len(cve_data)} CVE entries with metrics and saved to {output_file}.")


# Save the filtered DataFrame to an Excel file for merge
output_path = "../../2.Data_merge/Datasets/cve_data.xlsx"
df_filtered.to_excel(output_path, index=False)

print(f"Filtered data successfully saved to {output_path}")

Extracted 10 CVE entries with metrics and saved to cve_data.xlsx.
Filtered data successfully saved to ../../2.Data_merge/Datasets/cve_data.xlsx


In [8]:
df

,CVE ID,Description,Weaknesses,baseScore,baseSeverity,attackVector,attackComplexity,privilegesRequired,userInteraction,confidentialityImpact,integrityImpact,availabilityImpact,exploitabilityScore,CVSS Version
0,CVE-2014-5433,An unauthenticated remote attacker may be able...,CWE-255; CWE-312,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0
1,CVE-2020-12040,Sigma Spectrum Infusion System v's6.x (model 3...,CWE-319,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.1
2,CVE-2020-12047,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-259; CWE-798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.1
3,CVE-2014-5431,Baxter SIGMA Spectrum Infusion System version ...,CWE-259; CWE-798,6.8,MEDIUM,PHYSICAL,LOW,NONE,NONE,HIGH,HIGH,HIGH,0.9,3.0
4,CVE-2014-5434,Baxter SIGMA Spectrum Infusion System version ...,CWE-259; CWE-798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0
5,CVE-2020-12045,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-259; CWE-798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.1
6,CVE-2020-12039,Baxter Sigma Spectrum Infusion Pumps Sigma Spe...,CWE-259; CWE-798,2.4,LOW,PHYSICAL,LOW,NONE,NONE,LOW,NONE,NONE,0.9,3.1
7,CVE-2014-5432,Baxter SIGMA Spectrum Infusion System version ...,CWE-287; CWE-592,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0
8,CVE-2020-12041,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-732,9.4,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,LOW,HIGH,3.9,3.1
9,CVE-2020-12043,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-672,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.1
